# Design2Code Model Setup and Testing
## Qwen3-VL vs VLM_WebSight_finetuned

This notebook sets up both models and tests them on Design2Code tasks.

In [2]:
import transformers, torch, torchvision, torchaudio, flash_attn

In [3]:
# Run this once to install all dependencies
!pip install accelerate sentencepiece pillow
!pip install qwen-vl-utils  # Qwen specific utilities
!pip install datasets huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 14.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [accelerate]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.7/39.7 MB 29.5 MB/s  0:00:016m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [qwen-vl-utils]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 1.4 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 72.8 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 43.8 MB/s  0:00:006m0:00:01
  Attempting uninstall: huggingface_hub0m╺━━━━━━━━━━ 14/19 [multiprocess]
    Found existing installation: huggingface-hub 0.36.0━━━━━━━ 14/19 [multiprocess]
    Uninstalling huggingface-hub-0.36.0:0m╺━━━━━━━━━━ 14/19 [multiprocess]
      Successfully uninstalled huggingface-hub-0.36.0━━━━━━━━━ 14/19 [multiprocess]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19/19 [datasets]/19 [datasets]ce_hub]
ERROR: pip's dependency resolver does not currently

In [6]:
from PIL import Image
import requests
from io import BytesIO
from transformers import AutoProcessor, AutoModelForVision2Seq
from transformers import Qwen3VLForConditionalGeneration, AutoTokenizer
from qwen_vl_utils import process_vision_info

## Setup Qwen3-VL Model

In [21]:
import hf_transfer

In [23]:
qwen_model = Qwen3VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-2B-Instruct", dtype="auto", cache_dir="/root/"
)

qwen_processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-2B-Instruct", cache_dir="/root/")

In [27]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
qwen_model.to(DEVICE)
qwen_model.eval()

Using device: cuda


Qwen3VLForConditionalGeneration(
  (model): Qwen3VLModel(
    (visual): Qwen3VLVisionModel(
      (patch_embed): Qwen3VLVisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3VLVisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3VLVisionAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3VLVisionMLP(
            (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
          )
        )
      )
 

## Test Qwen3-VL

In [ ]:
from datasets import load_dataset
ds = load_dataset("SALT-NLP/Design2Code-hf", split="train")
sample = ds[0]   # pick the first item
image = sample["image"]

Generating train split: 100%|████████| 484/484 [00:00<00:00, 3335.05 examples/s]


dict_keys(['image', 'text'])


In [28]:
def generate_code_qwen(image_path, prompt="Convert this UI design into complete HTML and CSS code."):
    """
    Generate HTML/CSS code using Qwen3-VL-2B-Instruct

    Args:
        image_path: Path to image file or URL
        prompt: Instruction prompt for code generation

    Returns:
        Generated HTML/CSS code as string
    """
    # Load image
    if image_path.startswith('http'):
        response = requests.get(image_path)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image = Image.open(image_path).convert("RGB")

    # Prepare conversation in Qwen format
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt}
            ]
        }
    ]

    # Apply chat template
    text = qwen_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    # Process vision inputs
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = qwen_processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to(qwen_model.device)

    # Generate
    with torch.no_grad():
        output_ids = qwen_model.generate(
            **inputs,
            max_new_tokens=2048,
            temperature=0.7,
            do_sample=True
        )

    # Decode
    generated_text = qwen_processor.batch_decode(
        output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True
    )[0]

    return generated_text


# Use a web design screenshot for testing
test_image = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/car.jpg"

qwen_output = generate_code_qwen(test_image)
print("\n=== Qwen3-VL-2B Test Output ===")
print(qwen_output[:1000])  # Print first 1000 chars
print("\n... (output truncated)")
print(f"\nTotal output length: {len(qwen_output)} characters")


=== Qwen3-VL-2B Test Output ===
user
Convert this UI design into complete HTML and CSS code.
assistant
```html
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>Classic Car</title>
  <style>
    body {
      margin: 0;
      padding: 0;
      background-color: #f0f0f0;
      font-family: Arial, sans-serif;
    }

   .container {
      width: 100%;
      max-width: 1200px;
      margin: 0 auto;
      padding: 20px;
    }

   .car {
      position: relative;
      width: 100%;
      height: 400px;
      background-color: #000;
      border-radius: 10px;
      overflow: hidden;
      box-shadow: 0 0 10px rgba(0, 0, 0, 0.5);
      margin: 20px auto;
      border: 2px solid #000;
    }

   .car img {
      width: 100%;
      height: 100%;
      object-fit: cover;
    }

   .car.text {
      position: absolute;
      top: 50%;
      left: 50%;
      transform: translate(-50%, -50%);
      color: #fff;
      font-size: 24px;
      font-weight: bold;
      text-align: 

##  Setup VLM_WebSight_finetuned Model

In [17]:
from transformers import AutoModelForCausalLM, AutoProcessor

LOCAL_MODEL_DIR = "models--HuggingFaceM4--VLM_WebSight_finetuned/snapshots/a5c2b06bfee0bd713cf2a6b3e4d46f94dd8fe839/"

websight_processor = AutoProcessor.from_pretrained(
    LOCAL_MODEL_DIR,
)

websight_model = AutoModelForCausalLM.from_pretrained(
    LOCAL_MODEL_DIR,
    trust_remote_code=True,
    torch_dtype="auto",
    # device_map="auto",
    low_cpu_mem_usage=True,
)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
websight_model.to(DEVICE)
websight_model.eval()

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.
The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.
Loading checkpoint shards: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 24.49it/s]
The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.


Using device: cuda


VMistralForVisionText2Text(
  (model): VMistralModel(
    (embed_tokens): DecoupledEmbedding(
      num_embeddings=32000, num_additional_embeddings=2, embedding_dim=4096, partially_freeze=False
      (additional_embedding): Embedding(2, 4096)
    )
    (vision_model): SiglipVisionModel(
      (vision_model): SiglipVisionTransformer(
        (embeddings): SiglipVisionEmbeddings(
          (patch_embedding): Conv2d(3, 1152, kernel_size=(14, 14), stride=(14, 14), padding=valid)
          (position_embedding): Embedding(4624, 1152)
        )
        (encoder): SiglipEncoder(
          (layers): ModuleList(
            (0-26): 27 x SiglipEncoderLayer(
              (self_attn): SiglipFlashAttention2(
                (k_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (v_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (q_proj): Linear(in_features=1152, out_features=1152, bias=True)
                (out_proj): Linear(in_features=1152,

In [ ]:
# # --- load model & processor ---
# websight_processor = AutoProcessor.from_pretrained(
#     "HuggingFaceM4/VLM_WebSight_finetuned",
#     cache_dir="/root/"
# )

# websight_model = AutoModelForCausalLM.from_pretrained(
#     "HuggingFaceM4/VLM_WebSight_finetuned",
#     trust_remote_code=True,
#     torch_dtype="auto",        # lets HF pick fp16/bf16
#     device_map="auto",         # loads directly onto GPU if space allows
#     low_cpu_mem_usage=True,
#     cache_dir="/root/"
# )

In [18]:
import torch
from PIL import Image
from transformers.image_utils import to_numpy_array, PILImageResampling, ChannelDimension
from transformers.image_transforms import resize, to_channel_dimension_format
import requests

def convert_to_rgb(image):
    """Convert image to RGB, handling transparency properly"""
    if image.mode == "RGB":
        return image

    image_rgba = image.convert("RGBA")
    background = Image.new("RGBA", image_rgba.size, (255, 255, 255))
    alpha_composite = Image.alpha_composite(background, image_rgba)
    alpha_composite = alpha_composite.convert("RGB")
    return alpha_composite


def custom_transform(x):
    """
    Custom image transformation for WebSight model
    - Converts to RGB
    - Resizes to 960x960 with BILINEAR interpolation
    - Normalizes with model-specific mean/std
    """
    x = convert_to_rgb(x)
    x = to_numpy_array(x)
    x = resize(x, (960, 960), resample=PILImageResampling.BILINEAR)
    x = websight_processor.image_processor.rescale(x, scale=1 / 255)
    x = websight_processor.image_processor.normalize(
        x,
        mean=websight_processor.image_processor.image_mean,
        std=websight_processor.image_processor.image_std
    )
    x = to_channel_dimension_format(x, ChannelDimension.FIRST)
    x = torch.tensor(x)
    return x


def generate_code_websight(image_path, prompt="Convert this UI design into HTML/CSS.", max_length=512):
    DEVICE = next(websight_model.parameters()).device
    image_seq_len = websight_model.config.perceiver_config.resampler_n_latents
    BOS = websight_processor.tokenizer.bos_token
    bad_ids = websight_processor.tokenizer(["<image>", "<fake_token_around_image>"],
                                           add_special_tokens=False).input_ids

    # Load image
    if image_path.startswith('http'):
        img = Image.open(requests.get(image_path, stream=True).raw)
    else:
        img = Image.open(image_path)
    img = convert_to_rgb(img)
           
    # Transform
    pixel_values = custom_transform(img).unsqueeze(0).to(DEVICE)

    # Text input
    inputs = websight_processor.tokenizer(
        f"{BOS}<fake_token_around_image>{'<image>' * image_seq_len}<fake_token_around_image>",
        return_tensors="pt",
        add_special_tokens=False
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    inputs["pixel_values"] = pixel_values

    # Generate
    with torch.no_grad():
        ids = websight_model.generate(
            **inputs,
            bad_words_ids=bad_ids,
            max_length=max_length,
            use_cache=False  # ✅ prevents AttributeError
        )

    return websight_processor.batch_decode(ids, skip_special_tokens=True)[0]

# Skip FlashAttention adjustment (not applicable for WebSight)
print("Testing VLM_WebSight_finetuned...")

test_image = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/car.jpg"
websight_output = generate_code_websight(test_image)

print("\n=== WebSight Output ===")
print(websight_output[:1000])  # Print first 1000 chars
print("\n... (output truncated)")
print(f"\nTotal output length: {len(websight_output)} characters")

Testing VLM_WebSight_finetuned...

=== WebSight Output ===
<html>
<style>
body {
    font-family: Arial, sans-serif;
    margin: 0;
    padding: 0;
    background-color: #f2f2f2;
}

.container {
    width: 80%;
    margin: auto;
    overflow: hidden;
}

.carousel {
    width: 100%;
    height: 400px;
    position: relative;
}

.slide {
    width: 100%;
    height: 400px;
    position: absolute;
    top: 0;
    left: 0;
    opacity: 0;
    transition: opacity 1s;
}

.slide.active {
    opacity: 1;
}

.slide .image {
    width: 100%;
    height: 400px;
    background-color: #ccc;
}

.slide p {
    text-align: center;
    padding: 20px;
}
</style>
<body>
    <div class="container">
        <div class="carousel">
            <div class="slide">
                <div class="image" id="image1"></div>
                <p>Our latest model, the Ferrari, is a true revolutionary in the car market. It's the perfect blend of speed and luxury.</p>
            </div>
            <div class="slide">
   